<div style="background:linear-gradient(90deg,#1A2E4A 0%,#2E5E9B 100%);color:white;padding:20px 26px;border-radius:12px;">
<span style="font-size:12px;letter-spacing:3px;color:#D6E8F7;">CURSO IA APLICADA A CIENCIA DE MATERIALES &nbsp;·&nbsp; DÍA 5 · CIERRE</span><br>
<span style="font-size:26px;font-weight:700;">🧭 Más allá del curso</span><br>
<span style="font-size:14px;color:#D6E8F7;">¿Dónde hay datos de mi tipo de material, y cómo empiezo con ellos?</span>
</div>

| Bloque | Contenido | Tiempo |
|--------|-----------|--------|
| 1 | Mapa de datasets por tipo de material | 5 min |
| 2 | Los 45 datasets que trae matminer, buscados por palabra | 5 min |
| 3 | Demo: band gaps **medidos**, mismo pipeline del martes | 5 min |
| 4 | Demo: resistencia de aceros (necesita internet) | 3 min |
| 5 | Cómo traer su propio dataset y dónde seguir aprendiendo | 2 min |

**La idea:** todo lo que hicimos esta semana (descriptores, Random Forest, filtros, BO, visión)
funciona igual con otros datos. Lo que cambia es **de dónde salen los datos y qué tan confiables son**.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomas0821/ia-aplicada-ciencia-materiales/blob/main/notebooks/colab/07_mas_alla_del_curso.ipynb)

**Versión para Google Colab.** Corra primero la celda ☁️ de abajo: baja el material del curso desde GitHub e instala las librerías que Colab no trae. Después siga como en Kabré, de arriba hacia abajo.

In [ ]:
# ☁️ Preparar Google Colab: baja el material del curso e instala lo que falta.
# Córrala primero. Tarda de 1 a 4 minutos la primera vez; en Kabré no hace nada.
import os, sys, glob, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    CURSO_COLAB = "/content/ia-aplicada-ciencia-materiales"
    if not os.path.isdir(CURSO_COLAB):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/tomas0821/ia-aplicada-ciencia-materiales", CURSO_COLAB], check=True)
    os.chdir(CURSO_COLAB)
    for z, destino in [("mattergen_src.zip", "."), ("datos/mattergen_fallback.zip", "datos"),
                       ("datos/vision/sem_prueba.zip", "datos/vision")]:
        subprocess.run(["unzip", "-q", "-n", z, "-d", destino], check=True)
    for primera in glob.glob("datos/**/*.parte0", recursive=True):   # archivos grandes, subidos en trozos
        final = primera[: -len(".parte0")]
        if not os.path.exists(final):
            trozos = sorted(glob.glob(final + ".parte*"), key=lambda p: int(p.rsplit("parte", 1)[1]))
            with open(final, "wb") as salida:
                for t in trozos:
                    salida.write(open(t, "rb").read())
    paquetes = ["pymatgen==2026.5.4", "matminer==0.10.1"]
    if paquetes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquetes], check=True)
    os.chdir(CURSO_COLAB + "/notebooks")
    print("✓ Listo. Material del curso en", CURSO_COLAB)
else:
    print("No estamos en Colab: esta celda no hace nada.")

In [ ]:
import warnings, io, contextlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
AZUL, NARANJA, TINTA, GRIS = "#2E5E9B", "#D96F0F", "#1A2E4A", "#8A94A3"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.22, "axes.titleweight": "bold", "figure.dpi": 105})

def encontrar_datos() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents[:3]]:
        if (base / "datos").is_dir():
            return base / "datos"
    raise FileNotFoundError("No encuentro la carpeta datos/")

DATOS = encontrar_datos()
print("Datos del curso en:", DATOS)

---
<div style="background:#1A2E4A;color:white;padding:10px 18px;border-radius:8px;margin-top:14px;"><b>1 · Mapa de datasets por tipo de material</b> <span style="float:right;background:#E87C1E;color:white;padding:2px 12px;border-radius:12px;font-size:12px;font-weight:600;">⏱ 5 min</span></div>


Esta semana usamos sobre todo cristales inorgánicos calculados con DFT. Hay datos para casi cualquier familia de materiales:

| Familia | Dataset | Qué trae | Cómo se accede |
|---|---|---|---|
| Cristales inorgánicos (DFT) | [Materials Project](https://next-gen.materialsproject.org) | Estructuras, energías, gaps, elásticas | API con key gratis (`mp-api`, martes) |
| | [OQMD](https://oqmd.org) | Energías de formación y estabilidad | Descarga completa o API |
| | [JARVIS-DFT](https://jarvis.nist.gov) (NIST) | Más de 80 000 materiales, incluye 2D | Paquete `jarvis-tools`; 2 versiones ya están en matminer |
| | [Alexandria](https://alexandria.icams.rub.de) | Cristales relajados con PBE, PBEsol y SCAN, fonones | Descarga por bloques |
| | [NOMAD](https://nomad-lab.eu) | Repositorio de cálculos de muchos códigos | Web y API |
| Comparar modelos | [Matbench](https://matbench.materialsproject.org) | 13 tareas, de 312 a 132 752 muestras | Ya viene en matminer (bloque 2) |
| | [Matbench Discovery](https://matbench-discovery.materialsproject.org) | Ranking de modelos para encontrar cristales estables | Web |
| MOFs | [QMOF](https://github.com/Andrew-S-Rosen/QMOF) | Más de 20 000 MOFs con propiedades DFT | Figshare |
| Catálisis | [Open Catalyst OC20](https://arxiv.org/abs/2010.09990) | Adsorbatos sobre superficies (DFT) | Descarga grande |
| Polímeros | [PolyInfo](https://polymer.nims.go.jp/en/) (NIMS) | Propiedades medidas de polímeros | Web, registro gratis |
| Espectros | [RRUFF](https://www.rruff.net/) | Minerales: Raman, IR, química, fotos | Descarga por muestra |
| | [Open Specy](https://cran.r-project.org/package=OpenSpecy) | Bibliotecas FTIR y Raman de plásticos (Cowger et al. 2021) | Paquete de R y web |
| Micrografías | [UHCSDB](https://materialsdata.nist.gov/handle/11256/940) (NIST) | Micrografías de acero de ultra alto carbono | Descarga |
| | [NEU-DET en Roboflow](https://universe.roboflow.com/park-sung-ho/neu-det-classification) | 6 defectos en acero laminado (miércoles) | Roboflow, key gratis |
| Experimentales en matminer | `expt_gap_kingsbury`, `steel_strength`, `glass_*`, `superconductivity2018`… | Datos **medidos** | `load_dataset` (bloque 2) |

🤔 **Pregunta para el grupo:** ¿cuál de estas familias se parece más a lo que ustedes miden o fabrican?

---
<div style="background:#1A2E4A;color:white;padding:10px 18px;border-radius:8px;margin-top:14px;"><b>2 · Los 45 datasets de matminer</b> <span style="float:right;background:#E87C1E;color:white;padding:2px 12px;border-radius:12px;font-size:12px;font-weight:600;">⏱ 5 min</span></div>


`matminer` trae un catálogo con descripción, tamaño y columnas de cada dataset. **Leer el catálogo no necesita internet**;
solo bajar los datos.

In [ ]:
from matminer.datasets import (get_available_datasets, get_dataset_num_entries,
                               get_dataset_description, get_dataset_columns, load_dataset)

with contextlib.redirect_stdout(io.StringIO()):          # la función imprime todo; lo silenciamos
    nombres = get_available_datasets(print_format=None)

PALABRAS_MEDIDO = ("expt", "citrine", "steel", "glass", "superconductivity", "ucsb")
catalogo = pd.DataFrame({
    "dataset": nombres,
    "filas": [get_dataset_num_entries(n) for n in nombres],
    "origen": ["medido" if any(p in n for p in PALABRAS_MEDIDO) else "calculado" for n in nombres],
    "columnas": [", ".join(get_dataset_columns(n))[:70] for n in nombres],
}).sort_values("filas")

print(f"{len(catalogo)} datasets: {sum(catalogo.origen == 'medido')} con datos medidos (aprox.), el resto calculados")
catalogo.style.hide(axis="index")

<div style="background:#EEF3F9;border-left:6px solid #2E5E9B;padding:8px 14px;border-radius:6px;margin-top:8px;"><b style="color:#2E5E9B;">🔍 Cómo funciona este código</b></div>


- `get_available_datasets` da los nombres; `get_dataset_num_entries` y `get_dataset_columns` leen la ficha de cada uno sin bajar nada.
- La columna **origen** es una etiqueta aproximada hecha por el nombre (`expt` = experimental). Siempre confirmen leyendo la descripción.
- Fíjense en el tamaño: los datasets **medidos** tienen cientos o pocos miles de filas; los **calculados**, decenas o cientos de miles.

In [ ]:
def buscar(palabra):
    """Muestra los datasets cuya descripción menciona la palabra (en inglés)."""
    for n in nombres:
        texto = get_dataset_description(n)
        if palabra.lower() in texto.lower():
            print(f"▶ {n}  ({get_dataset_num_entries(n)} filas)\n   {texto[:220]}...\n")

buscar("thermal")       # prueben: "steel", "glass", "superconduct", "dielectric", "perovskite", "2D"

---
<div style="background:#1A2E4A;color:white;padding:10px 18px;border-radius:8px;margin-top:14px;"><b>3 · Demo: band gaps medidos, mismo pipeline del martes</b> <span style="float:right;background:#E87C1E;color:white;padding:2px 12px;border-radius:12px;font-size:12px;font-weight:600;">⏱ 5 min</span></div>


El martes predijimos el módulo de compresibilidad **calculado**. Ahora el **band gap medido** en laboratorio
(`matbench_expt_gap`, 4 604 composiciones). Viene en la carpeta del curso, así que no necesita internet.

In [ ]:
gaps = load_dataset("matbench_expt_gap", data_home=str(DATOS / "matminer_cache"))
gaps = gaps.rename(columns={"gap expt": "gap_eV"})

print(f"{len(gaps)} materiales; {100 * (gaps.gap_eV == 0).mean():.0f}% son metales (gap = 0)")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(gaps.gap_eV[gaps.gap_eV > 0], bins=50, color=AZUL)
ax.set_xlabel("Band gap medido (eV), sin contar metales"); ax.set_ylabel("Materiales")
ax.set_title("Band gaps experimentales"); plt.tight_layout(); plt.show()
gaps.sort_values("gap_eV").tail(5)

In [ ]:
# Mismo pipeline del martes: composición -> 132 descriptores Magpie -> Random Forest
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict, KFold

magpie = ElementProperty.from_preset("magpie")
gaps["comp"] = gaps["composition"].map(Composition)
X = magpie.featurize_dataframe(gaps, "comp", ignore_errors=True, pbar=False)[magpie.feature_labels()].fillna(0)
y = gaps.gap_eV.values

# Validación cruzada de 5 partes: cada material se predice con un modelo que no lo vio (~30 s)
pred = cross_val_predict(RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0),
                         X, y, cv=KFold(5, shuffle=True, random_state=0))

mae, mae_base = np.abs(pred - y).mean(), np.abs(y - y.mean()).mean()
print(f"Error medio del modelo: {mae:.2f} eV   |   adivinar siempre el promedio: {mae_base:.2f} eV")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y, pred, s=6, alpha=0.4, color=AZUL)
ax.plot([0, 12], [0, 12], color=TINTA, ls=":")
ax.set_xlabel("Gap medido (eV)"); ax.set_ylabel("Gap predicho (eV)")
ax.set_title("Solo con la composición"); plt.tight_layout(); plt.show()

<div style="background:#EEF3F9;border-left:6px solid #2E5E9B;padding:8px 14px;border-radius:6px;margin-top:8px;"><b style="color:#2E5E9B;">💬 Qué decir</b></div>


- Es **el mismo código** del martes; solo cambió la columna objetivo. Eso es lo que se llevan.
- El modelo mejora mucho sobre adivinar el promedio, pero se equivoca casi medio eV: la composición no
  sabe de estructura (los polimorfos tienen la misma fórmula y distinto gap), y los datos medidos traen ruido.
- Miren la nube pegada al eje vertical: metales (gap 0) que el modelo cree semiconductores, y al revés.
  Muchas veces conviene **primero clasificar** metal o no metal (`matbench_expt_is_metal`) y después predecir el gap.

---
<div style="background:#1A2E4A;color:white;padding:10px 18px;border-radius:8px;margin-top:14px;"><b>4 · Demo: resistencia de aceros (necesita internet)</b> <span style="float:right;background:#E87C1E;color:white;padding:2px 12px;border-radius:12px;font-size:12px;font-weight:600;">⏱ 3 min</span></div>


`matbench_steels`: 312 aceros con su **límite elástico medido** y su composición. Si el nodo no tiene internet,
la celda lo avisa y siguen.

In [ ]:
CACHE = Path.home() / "matminer_datos"          # carpeta propia: la del paquete puede ser de solo lectura
CACHE.mkdir(exist_ok=True)
try:
    aceros = load_dataset("matbench_steels", data_home=str(CACHE))
except Exception as e:
    aceros = None
    print(f"No se pudo bajar ({type(e).__name__}). Sin internet en este nodo: sigan al bloque 5.")

if aceros is not None:
    aceros["comp"] = aceros["composition"].map(Composition)
    Xa = magpie.featurize_dataframe(aceros, "comp", ignore_errors=True, pbar=False)[magpie.feature_labels()].fillna(0)
    ya = aceros["yield strength"].values
    pa = cross_val_predict(RandomForestRegressor(n_estimators=200, random_state=0), Xa, ya,
                           cv=KFold(5, shuffle=True, random_state=0))
    print(f"{len(aceros)} aceros. Error medio: {np.abs(pa - ya).mean():.0f} MPa  |  "
          f"adivinar el promedio: {np.abs(ya - ya.mean()).mean():.0f} MPa")
    print(aceros.sort_values("yield strength").tail(3)[["composition", "yield strength"]])

<div style="background:#EEF3F9;border-left:6px solid #2E5E9B;padding:8px 14px;border-radius:6px;margin-top:8px;"><b style="color:#2E5E9B;">💬 Qué decir</b></div>


- 312 filas es **small data**, como casi todo lo experimental (mañana lo vemos en el Brief B).
- La composición sola no sabe del **tratamiento térmico** ni del laminado, que cambian mucho la resistencia.
  Para mejorar habría que agregar columnas de proceso, como el Día 4.

---
<div style="background:#1A2E4A;color:white;padding:10px 18px;border-radius:8px;margin-top:14px;"><b>5 · Su propio dataset y dónde seguir</b> <span style="float:right;background:#E87C1E;color:white;padding:2px 12px;border-radius:12px;font-size:12px;font-weight:600;">⏱ 2 min</span></div>


**Antes de entrenar con datos propios, revisen:**

1. **Una columna objetivo clara**, con unidades y medida igual en todas las filas.
2. **¿Cuántas filas?** Menos de unas cientos: modelos simples, validación cruzada y BO, como el Brief B.
3. **¿Medido o calculado?** Lo medido trae ruido; repetir mediciones ayuda a saber cuánto.
4. **Duplicados por fórmula:** si la misma fórmula aparece varias veces, sepárenlas por fórmula al validar
   (`GroupKFold`), o el modelo hace trampa.
5. **Licencia y origen:** ¿se puede publicar con esos datos?

**Plantilla mínima** (cambien el nombre del archivo y de las columnas):

```python
mis = pd.read_csv("mis_datos.csv")                       # una fila por muestra
mis["comp"] = mis["formula"].map(Composition)            # columna con la fórmula
Xm = magpie.featurize_dataframe(mis, "comp", ignore_errors=True)[magpie.feature_labels()]
# ... y de aquí en adelante, igual que el notebook 02
```

**Notebooks para seguir aprendiendo:**

| Dónde | Qué hay |
|---|---|
| [Ejemplos de matminer](https://nbviewer.org/github/hackingmaterials/matminer_examples/blob/main/matminer_examples/index.ipynb) | Bajar datos, descriptores, visualizar, entrenar e interpretar |
| [Matbench](https://matbench.materialsproject.org) | Cómo comparar su modelo con los publicados |
| `soluciones/` del curso | Soluciones de los días 1 a 4 y de los dos briefs |
| Notebook 06, al final | Cómo bajar un dataset de Roboflow Universe |

<div style="background:#FFF3E6;border-left:6px solid #E87C1E;padding:8px 14px;border-radius:6px;margin-top:8px;"><b style="color:#C25E08;">🎯 Mini reto · 3 min</b></div>


Con `buscar()` encuentren **un dataset de matminer cercano a su trabajo**. Anoten: nombre, cuántas filas,
cuál sería la columna objetivo y si es medido o calculado. Si tienen internet, cárguenlo con
`load_dataset("nombre", data_home=str(CACHE))` y miren `.head()`.

In [ ]:
# [Su código aquí]